# T04 · Théorie des probabilités — ADIF72

**Objectifs :**
- Espace probabiliste et axiomes de Kolmogorov
- Variables aléatoires discrètes et continues
- Lois usuelles : binomiale, poisson, normale, bêta, exponentielle
- Espérance, variance, écart-type
- Théorème de Bayes
- Modélisation d'incertitude (Monte-Carlo, VaR)

**Pourquoi c'est central :** la probabilité est le **pont** entre la
géométrie (T01-T02), le calcul (T03) et la statistique (T05-T06).
Toute inférence repose sur le modèle de probabilité qu'on adopte.

In [ ]:
import numpy as np
from scipy import stats
np.set_printoptions(precision=4, suppress=True, linewidth=100)
print("Prêt. scipy", stats.norm.cdf(0))

## 1 · Espace probabiliste & axiomes de Kolmogorov
Un **espace probabiliste** est un triplet `(Ω, F, P)` :
- `Ω` : ensemble des résultats possibles (espace fondamental)
- `F` : ensemble des événements (σ-algèbre)
- `P` : fonction de probabilité

**Axiomes :**
1. `P(A) ≥ 0` pour tout événement `A`
2. `P(Ω) = 1`
3. Pour des événements **disjoints** `Aᵢ` : `P(∪ Aᵢ) = Σ P(Aᵢ)`

Ces 3 axiomes suffisent à dériver **toute** la théorie classique.
En pratique, on définit directement `P` sur les éléments de base et on
propage par additivité.

In [ ]:
# Dé 6 faces équilibré
sample_space = {i: 1/6 for i in range(1, 7)}
assert np.isclose(sum(sample_space.values()), 1.0), "P(Ω)=1"
assert all(v >= 0 for v in sample_space.values()), "P≥0"
P_impair = sum(sample_space[i] for i in [1, 3, 5])
print(f"P({{1,3,5}}) = {P_impair} = 3/6 (additivité)")
assert np.isclose(P_impair, 0.5)

### Exercice corrigé

In [ ]:
# ---- Exercice P1 : P(face paire ET pile) ----
P = (3/6) * (1/2)
print(f"P1-ex : P(paire et pile) = {P} = 1/4 ( independance )")
assert np.isclose(P, 0.25)


## 2 · Variables aléatoires (discrètes et continues)
Une **v.a. discrète** `X` prend une valeur finie ou dénombrable. Sa
**distribution** (PMF) : `P(X = k)`.
Conditions : `P(X=k) ≥ 0` et `Σ_k P(X=k) = 1`.
Moments :
```
E[X]   = Σ k · P(X=k)
E[X²]  = Σ k² · P(X=k)
Var[X] = E[X²] - (E[X])²
```
Exemple : nombre de piles sur 4 lancers = **Binomiale(n=4, p=0.5)**.

In [ ]:
n, p = 4, 0.5
k = np.arange(n + 1)
pmf = stats.binom.pmf(k, n, p)
for kv, pv in zip(k, pmf):
    print(f"P(X={kv}) = {pv:.4f}")
print("Somme =", pmf.sum())
print(f"E[X] = np = {n*p:.2f}   Var[X] = np(1-p) = {n*p*(1-p):.2f}")
assert np.isclose(pmf.sum(), 1.0)

### Variable aléatoire continue — Normal

Une **v.a. continue** `X` a une **densité** `f(x)` (pas une probabilité
directe : on intègre pour obtenir des probas).
`∫ f(x) dx = 1`.
Pour `X ~ N(μ, σ²)` :
- `E[X] = μ`, `Var[X] = σ²`
- `P(a < X < b) = Φ((b-μ)/σ) - Φ((a-μ)/σ)`
- **Règle 68-95-99.7** : `P(|Z|<1)=68%`, `<2)=95%`, `<3)=99.7%`

C'est la distribution **fondamentale** — le TCL (T05) garantit que
presque toute moyenne d'échantillon est quasi normale, même si la
population ne l'est pas.

In [ ]:
mu, sigma = 2.0, 1.5
print(f"X ~ N({mu}, {sigma}²)")
print(f"E[X] = {mu}   Var[X] = {sigma**2}   sd = {sigma}")
c = stats.norm.cdf(3, mu, sigma) - stats.norm.cdf(0, mu, sigma)
print(f"P(0<X<3) = {c:.4f}")
q05 = stats.norm.ppf(0.05, mu, sigma); q95 = stats.norm.ppf(0.95, mu, sigma)
print(f"5e percentile = {q05:.3f}, 95e = {q95:.3f}")
for k_, pct in [(1, 68.27), (2, 95.45), (3, 99.73)]:
    val = 2 * stats.norm.cdf(k_) - 1
    print(f"P(|Z|<{k_}) = {val:.4f}  (~{pct}%)")

### Exercice corrigé

In [ ]:
# ---- Exercice P2 : E[X], Var[X] d'une v.a. discrete ; P(0<=Z<=1) ----
vals = np.array([0.0, 1, 2, 3]); pr = np.array([0.1, 0.2, 0.5, 0.2])
E = (vals*pr).sum(); V = ((vals-E)**2*pr).sum()
print(f"P2-ex : E[X] = {E}, Var[X] = {V:.3f}")
assert np.isclose(E, 1.8) and np.isclose(V, 0.76)
p_z = stats.norm.cdf(1) - 0.5
print(f"       P(0 <= Z <= 1) = {p_z:.4f}")
assert np.isclose(p_z, 0.3413, atol=1e-4)


## 3 · Lois usuelles en science des données
| Loi | Paramètres | Usage typique |
|---|---|---|
| **Binomiale** | n, p | Succès/échecs (n essais) |
| **Poisson** | λ | Événements rares (appels, défauts) |
| **Normale** | μ, σ² | Données naturelles, moyenne (TCL) |
| **Exponentielle** | λ | Temps entre événements |
| **Bêta** | α, β | Proportions / **prior bayésien** |
| **LogNormale** | μ, σ | Prix, tailles, revenus (>0, asym) |

**Choix de loi** = première décision en modélisation. La loi détermine
les moments (E, Var), les tests statistiques, et l'inférence possible.

In [ ]:
lam = 3.0
print(f"Poisson(λ={lam}) : P(X=5) = {stats.poisson.pmf(5, lam):.4f}  E={lam} Var={lam}")
print(f"Binomiale(n=20,p=0.3) : P(X=5) = {stats.binom.pmf(5,20,0.3):.4f}")
print(f"Bernoulli(p=0.7) : E={0.7}, Var={0.7*0.3:.4f}")
print(f"Exponentielle(λ=2) : P(X<0.5) = {stats.expon.cdf(0.5, scale=0.5):.4f}")
print(f"Bêta(2,5) : E={2/7:.3f}, Var={2*5/(7**2*8):.4f}")

### Exercice corrigé

In [ ]:
# ---- Exercice P3 : P(Bin(10,0.1)>=1), P(Pois(1)>=1), E[Geom] ----
pB = 1 - stats.binom.pmf(0, 10, 0.1)
pP = 1 - stats.poisson.pmf(0, 1.0)
print(f"P3-ex : P(Bin(10,0.1)>=1) = {pB:.4f} ; P(Pois(1)>=1) = {pP:.4f}")
print(f"       E[Geom(0.5)] = {stats.geom.mean(0.5)}  (attendu : 2)")
assert abs(pB - 0.6513) < 1e-3 and abs(pP - 0.6321) < 1e-3


## 4 · Bayes & test d'hypothèse
**Formule** : pour des événements `A, B` avec `P(B) > 0` :
```
P(A|B) = P(B|A) P(A) / P(B)
```
- `P(A)` : **prior** (avant observation)
- `P(B|A)` : **vraisemblance** (comment B se présente si A est vrai)
- `P(B)` : **évidence** (normalisation)
- `P(A|B)` : **posterior** (après observation)

### L'exemple du test de dépistage (paradoxe du taux de base)
Un test a 99% de sensibilité et 5% de faux positifs. Prévalence = 1%.
→ Quelle est `P(malade | positif)` ?
Réponse contre-intuitive : **~17%**. La rareté de la maladie domine.
C'est un piège **très fréquent** en examen et en pratique.

In [ ]:
P_malade = 0.01
P_pos_malade = 0.99    # sensibilité
P_pos_sain = 0.05      # faux positifs
P_pos = P_pos_malade * P_malade + P_pos_sain * (1 - P_malade)
P_malade_pos = P_pos_malade * P_malade / P_pos
print(f"P(Positif) = {P_pos:.5f}")
print(f"P(Malade | Positif) = {P_malade_pos:.5f}  ({P_malade_pos*100:.2f}%)")
print("-> malgré 99% de sensibilité, très peu de malades (taux de base)")
assert P_malade_pos < 0.2

### Bayes avec prior Bêta (conjugaison Bêta-Binomiale)
Si `p ~ Bêta(α, β)` et qu'on observe `k` succès sur `n` essais,
alors `p | données ~ Bêta(α+k, β+n-k)`. C'est la forme la plus simple
de l'inférence bayésienne — on la retrouve partout (A/B testing,
modèles binaires).

In [ ]:
a, b = 1, 1   # prior uniforme
successes, trials = 3, 100
a_post, b_post = a + successes, b + trials - successes
print(f"Prior Bêta({a},{b}) + {successes} succès sur {trials}")
print(f"-> Posterior Bêta({a_post},{b_post})")
print(f"   E[prior]={a/(a+b):.3f} -> E[posterior]={a_post/(a_post+b_post):.3f}")
print(f"   Var[posterior]={a_post*b_post/((a_post+b_post)**2*(a_post+b_post+1)):.5f}")

### Test d'hypothèse (aperçu — détaillé en T06)

Le test répond à : « Les données contredisent-elles l'hypothèse nulle ? »
On calcule un **p-value** = P(observer un résultat aussi extrême
que celui observé **si H0 était vraie**).
- `p < α` (souvent 0.05) → rejeter H0
- `p ≥ α` → pas assez de preuve pour rejeter

Exemple : 60 piles sur 100 lancers. Est-ce une pièce équilibrée ?

In [ ]:
n, k, p0 = 100, 60, 0.5
z = (k - n*p0) / np.sqrt(n*p0*(1-p0))   # approx. normale
p_value = 2 * (1 - stats.norm.cdf(abs(z)))
print(f"z = {z:.3f}, p-value = {p_value:.5f}")
alpha = 0.05
print(f"A α={alpha}: {'REJET' if p_value < alpha else 'non-rejet'} de H0 (pièce équilibrée)")
assert p_value < 0.05

### Exercice corrigé

In [ ]:
# ---- Exercice P4 : Bayes (test de depistage) + Poisson ----
Ppos = 0.99*0.01 + 0.05*0.99
Pmp = 0.0099 / Ppos
print(f"P4-ex : P(+) = {Ppos:.5f}, P(M|+) = {Pmp:.5f}  (paradoxe du taux de base)")
assert Pmp < 0.2
p3 = stats.poisson.pmf(3, 2.5)
print(f"       Poisson(2.5) : P(X=3) = {p3:.4f} (= e^-2.5 * 2.5^3 / 6)")
assert np.isclose(p3, np.exp(-2.5) * 2.5**3 / 6)


## 5 · Monte-Carlo & familles exponentielles *(MML ch. 6.6)*
**Monte-Carlo** : approximer une intégrale / probabilité par simulation.
Utilisé quand la solution analytique est complexe (modèles financiers,
physiques, réseaux de neurones).
Convergence : l'erreur diminue en `O(1/√N)` — pour diviser par 10 l'erreur,
il faut 100× plus de simulations.

In [ ]:
rng = np.random.default_rng(7)
mu_l, sigma_l = 4.5, 0.5
N = 200_000
X = rng.lognormal(mu_l, sigma_l, N)
p_mc = np.mean(X > 100)
p_exact = 1 - stats.lognorm.cdf(100, s=sigma_l, scale=np.exp(mu_l))
print(f"P(X>100) : Monte-Carlo = {p_mc:.5f}, exacte = {p_exact:.5f}")
assert abs(p_mc - p_exact) < 0.01
var95 = np.percentile(X, 95)
print(f"VaR 95% = {var95:.3f}")

### Familles exponentielles & conjugaison *(MML ch. 6.6)*

Une loi est dans la **famille exponentielle** si sa densité s'écrit
```
p(x | θ) = h(x) · exp( η(θ)ᵀ T(x) - A(θ) )
```
avec `η` **paramètre naturel**, `T` **statistique suffisante**, `A`
fonction de normalisation. Les lois de base (Bernoulli, Gaussienne,
Poisson, Gamma...) sont toutes dans cette famille.

**Propriété clé — conjugaison** : si le prior `p(θ)` et la vraisemblance
`p(x|θ)` sont dans la même famille exponentielle (conjugués), alors le
postérieur est de la **même famille**, avec mise à jour analytique des
paramètres. C'est ce qui rend le bayésien **calculable** en pratique.

| Vraisemblance | Prior conjugué | Posterior |
|---|---|---|
| Bernoulli | Bêta | Bêta |
| Binomiale | Bêta | Bêta |
| Gaussienne (σ connu) | Gaussienne | Gaussienne |
| Poisson | Gamma | Gamma |
| Exponentielle | Gamma | Gamma |
| Gamma | Gamma | Gamma |

In [ ]:
# Conjugaison Poisson-Gamma : λ ~ Gamma(α, β), x ~ Poisson(λ)
# -> λ | x ~ Gamma(α + Σx, β + n)
from scipy import stats
rng = np.random.default_rng(1)
alpha, beta = 2.0, 1.0     # prior Gamma(shape=2, rate=1), E[λ]=2
lam_true = 5.0
x = rng.poisson(lam_true, 6)
print("Observations Poisson(5):", x)
print(f"E[λ prior] = {alpha/beta:.3f}")
print(f"MLE de λ   = {x.mean():.3f}")
print(f"E[λ post]  = {(alpha + x.sum()) / (beta + len(x)):.3f}")

# Le postérieur s'approche du MLE quand n grandit
for n_obs in [6, 60, 600]:
    xo = rng.poisson(lam_true, n_obs)
    e_post = (alpha + xo.sum()) / (beta + n_obs)
    print(f"  n={n_obs:4d} : E[λ post] = {e_post:.3f}  (MLE = {xo.mean():.3f})")

# Vérif par simulation du postérieur
post = stats.gamma(alpha + x.sum(), scale=1.0 / (beta + len(x)))
print(f"\nPosterior Gamma({alpha + x.sum():.0f}, rate={beta + len(x):.0f}) : "
      f"E={post.mean():.3f}, sd={post.std():.3f}")
assert np.isclose(post.mean(), (alpha + x.sum()) / (beta + len(x)))

### Exercice corrigé

In [ ]:
# ---- Exercice P5 : Monte-Carlo P(|Z| < 1) ----
rng_ = np.random.default_rng(11)
z = rng_.normal(0, 1, 200000)
emp = np.mean(np.abs(z) < 1)
print(f"P5-ex : P(|Z|<1) empirique = {emp:.4f}  (theorique = 0.6826)")
assert abs(emp - 0.6826) < 0.01


## 6 · Changement de variable & échantillonnage inverse *(MML ch. 6.7)*
Si `Y = g(X)` est une bijection, la densité de `Y` :
```
f_Y(y) = f_X(g⁻¹(y)) · |d/dy g⁻¹(y)|
```
Le **jacobien** (dét. de la matrice des dérivées) est l'analogue
multivarié du terme `|g⁻¹'(y)|`.

Applications :
- **échantillonnage par transformation inverse** : si `X ~ U(0,1)`
  alors `Y = F_Y⁻¹(X)` suit la loi de densité `F_Y`
- **normalisation de volume** en transformations (NFT, flow)
- **dérivation de lois** : `Y = -ln(U)` avec `U ~ U(0,1)` donne
  `Y ~ Exponentielle(1)

In [ ]:
# Y = -ln(X) pour X ~ U(0,1)  ->  Y ~ Exponentielle(1)
rng = np.random.default_rng(7)
X = rng.uniform(0, 1, 200000)
Y = -np.log(X)
ys = np.linspace(0, 5, 50)
hist, edges = np.histogram(Y, bins=ys, density=True)
mid = (edges[:-1] + edges[1:]) / 2
err = np.max(np.abs(hist - np.exp(-mid)))
print(f"Max |densité empirique - exp(-y)| = {err:.4f}  (Y ~ Exponentielle(1))")
assert err < 0.03

# Échantillonnage inverse pour N(μ, σ²) : y = μ + σ Φ⁻¹(u), u ~ U(0,1)
Z = stats.norm.ppf(rng.uniform(0, 1, 100000))
print(f"N(0,1) par transformation inverse : mean={Z.mean():.3f}, sd={Z.std():.3f}")
assert abs(Z.mean()) < 0.05 and abs(Z.std() - 1) < 0.05

# Multivarié : jacobien pour la transformation polaire
# (x,y) -> (r,θ) : |J| = r  ->  si (x,y) ~ N(0,I) alors r ~ Rayleigh
xy = rng.normal(0, 1, (100000, 2))
r = np.linalg.norm(xy, axis=1)
# densité de r : f(r) = r·exp(-r²/2)
rs = np.linspace(0, 4, 40)
hist_r, edges_r = np.histogram(r, bins=rs, density=True)
mid_r = (edges_r[:-1] + edges_r[1:]) / 2
err_r = np.max(np.abs(hist_r - mid_r * np.exp(-mid_r**2 / 2)))
print(f"Rayleigh : max err densité = {err_r:.4f}")
assert err_r < 0.03

### Exercice corrigé

In [ ]:
# ---- Exercice P6 : Y = -ln(X), X~U(0,1) -> Y~Exp(1) ----
rng_ = np.random.default_rng(7)
X = rng_.uniform(0, 1, 200000)
Y = -np.log(X)
p_emp = np.mean(Y < 1)
print(f"P6-ex : P(Y<1) = {p_emp:.4f}  (theorique = 1-e^-1 = {1-np.exp(-1):.4f})")
assert abs(p_emp - (1 - np.exp(-1))) < 0.01


## 7 · Synthèse — exercices intégrés

### Exercice corrigé

In [ ]:
# ---- Exercice P7 : moments de Bernoulli(p=0.7) ----
p = 0.7
print(f"P7-ex : E = {p}, Var = {p*(1-p):.2f}")
assert np.isclose(p*(1-p), 0.21)
print("\n✅ T04 termine — 7 points, exercices integres.")
